# 00 · Colab 환경 설정

**이 노트북을 가장 먼저 실행하세요.**\
Drive 마운트 → 저장소 클론 → 의존성 설치 → 환경 검증을 한 번에 수행합니다.

> 런타임 재시작 후에도 이 노트북을 다시 실행하면 됩니다.  
> Drive 마운트가 유지되는 경우 ③ 클론 셀은 건너뛰어도 됩니다.

## ① 파라미터 — 필요한 경우만 수정

In [ ]:
from pathlib import Path

# ── 수정 가능한 파라미터 ────────────────────────────────────────────
REPO_URL    = globals().get('REPO_URL',    'https://github.com/DaehyunY00/Military_Object_Detection.git')
REPO_BRANCH = globals().get('REPO_BRANCH', 'main')
REPO_DIR    = Path(globals().get('REPO_DIR',    '/content/Military_Object_Detection'))
DRIVE_MOUNT_POINT      = Path(globals().get('DRIVE_MOUNT_POINT', '/content/drive'))
WORKSPACE_ROOT         = Path(globals().get('WORKSPACE_ROOT',
                               str(DRIVE_MOUNT_POINT / 'MyDrive' / 'Military_Object_Detection')))
INSTALL_AUGMENTATION_DEPS = bool(globals().get('INSTALL_AUGMENTATION_DEPS', False))
FORCE_REMOUNT_DRIVE       = bool(globals().get('FORCE_REMOUNT_DRIVE', False))
# ──────────────────────────────────────────────────────────────────

print(f'REPO_DIR       : {REPO_DIR}')
print(f'WORKSPACE_ROOT : {WORKSPACE_ROOT}')
print(f'AUGMENTATION   : {INSTALL_AUGMENTATION_DEPS}')

## ② Google Drive 마운트

In [ ]:
try:
    from google.colab import drive
except ImportError as exc:
    raise RuntimeError('이 노트북은 Google Colab에서 실행해야 합니다.') from exc

drive.mount(str(DRIVE_MOUNT_POINT), force_remount=FORCE_REMOUNT_DRIVE)

for subdir in ['data', 'experiments', 'artifacts', 'checkpoints', 'logs']:
    (WORKSPACE_ROOT / subdir).mkdir(parents=True, exist_ok=True)

print(f'✅ Drive 마운트 완료 : {DRIVE_MOUNT_POINT}')
print(f'✅ Workspace 준비   : {WORKSPACE_ROOT}')

## ③ 저장소 클론 / 업데이트

> 이미 클론된 경우 최신 코드로 pull합니다.

In [ ]:
import os, subprocess

if REPO_DIR.exists() and (REPO_DIR / '.git').exists():
    subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', '--depth', '1', 'origin', REPO_BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'checkout', REPO_BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only', 'origin', REPO_BRANCH], check=True)
    print(f'✅ 저장소 업데이트 완료: {REPO_DIR}')
else:
    subprocess.run(
        ['git', 'clone', '--depth', '1', '--branch', REPO_BRANCH, REPO_URL, str(REPO_DIR)],
        check=True
    )
    print(f'✅ 저장소 클론 완료: {REPO_DIR}')

os.chdir(REPO_DIR)
print(f'cwd: {os.getcwd()}')

## ④ 의존성 설치

In [ ]:
import sys, subprocess

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-U', 'pip'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                '-r', str(REPO_DIR / 'requirements-colab.txt')], check=True)
print('✅ 기본 의존성 설치 완료')

if INSTALL_AUGMENTATION_DEPS:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    '-r', str(REPO_DIR / 'requirements-augmentation.txt')], check=True)
    print('✅ 증강 의존성 설치 완료 (diffusers, transformers ...)')

## ⑤ 환경 초기화 및 검증

In [ ]:
import sys
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

from mad.colab_utils import setup_colab_env, check_gpu

env = setup_colab_env(REPO_DIR, WORKSPACE_ROOT)
gpu_info = check_gpu(require=False)

print()
print('다음 단계: notebooks/01_prepare_dataset.ipynb')

## 다음 단계

설정 완료 후 **`notebooks/01_prepare_dataset.ipynb`** 를 열어  
데이터셋 파일을 `WORKSPACE_ROOT/data/` 아래에 업로드한 후 실행하세요.

### 런타임 재시작 후 빠른 재설정 (다른 노트북 첫 셀에 붙여넣기)
```python
import sys, os
REPO_DIR       = '/content/Military_Object_Detection'
WORKSPACE_ROOT = '/content/drive/MyDrive/Military_Object_Detection'
sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)
from mad.colab_utils import setup_colab_env
setup_colab_env(REPO_DIR, WORKSPACE_ROOT)
```